In [1]:
from sage.all import sigma, nth_prime
from sage.arith.all import moebius
import pickle
import ast

print(nth_prime(400))

2741


In [1]:
def get_all_newforms_at_level(level, expansion_terms=30, output_file=None):
    """
    Get ALL weight 2 rational newforms at a specific level.
    Only takes one curve per isogeny class (since they give the same newform).
    """
    from sage.databases.cremona import CremonaDatabase
    import time
    
    start_time = time.time()
    db = CremonaDatabase()
    
    rational_forms = []
    
    print(f"Weight 2 rational newforms at level {level}")
    print(f"Computing {expansion_terms} terms\n")
    
    # Open output file if specified
    if output_file:
        f_out = open(output_file, 'w')
    
    try:
        curves = db.allcurves(level)
        
        if curves:
            # Group by isogeny class (the letter part)
            isogeny_classes = {}
            for label in curves.keys():
                # Extract the letter (isogeny class identifier)
                letter = ''.join([c for c in label if c.isalpha()])
                if letter not in isogeny_classes:
                    isogeny_classes[letter] = label
            
            print(f"Found {len(isogeny_classes)} distinct newforms (isogeny classes)\n")
            
            # Process one curve per isogeny class
            for letter in sorted(isogeny_classes.keys()):
                label = isogeny_classes[letter]
                full_label = f"{level}{label}"
                E = EllipticCurve(full_label)
                rational_forms.append((level, 2, E))
                
                # Print header to console
                print(f"\n{'='*70}")
                print(f"Newform {len(rational_forms)}: Isogeny class {level}{letter}")
                print(f"Representative curve: {full_label}")
                print(f"{'='*70}")
                
                # Get coefficients
                an_list = E.anlist(expansion_terms + 1)
                coeffs = an_list[1:]
                
                # Write to file
                if output_file:
                    f_out.write(f"Isogeny class: {level}{letter}\n")
                    f_out.write(f"Representative curve: {full_label}\n")
                    f_out.write(f"Coefficients a_1 to a_{expansion_terms}:\n")
                    f_out.write(str(coeffs) + "\n\n")
                
                # Show first 10 coefficients to console
                print(f"Coefficients (first 10 of {expansion_terms}):")
                print(f"  a_1-a_10: {coeffs[:10]}")
                
                # Additional info
                print(f"\nCurve info:")
                print(f"  Conductor: {E.conductor()}")
                print(f"  Rank: {E.rank()}")
                
                if output_file:
                    f_out.write(f"Conductor: {E.conductor()}\n")
                    f_out.write(f"Rank: {E.rank()}\n")
                    f_out.write(f"{'='*70}\n\n")
    
    except Exception as e:
        print(f"Error at level {level}: {e}")
    
    if output_file:
        f_out.close()
        print(f"\nFull data written to: {output_file}")
    
    elapsed = time.time() - start_time
    print(f"\n{'='*70}")
    print(f"Found {len(rational_forms)} distinct newforms in {elapsed:.2f} seconds")
    print(f"{'='*70}")
    
    return rational_forms


results = get_all_newforms_at_level(
    level=27,
    expansion_terms=2800,
    output_file="/Users/barrybrent/data2/run3mar26no4.txt"
)

Weight 2 rational newforms at level 27
Computing 2800 terms

Found 1 distinct newforms (isogeny classes)


Newform 1: Isogeny class 27a
Representative curve: 27a1
Coefficients (first 10 of 2800):
  a_1-a_10: [1, 0, 0, -2, 0, 0, -1, 0, 0, 0]

Curve info:
  Conductor: 27
  Rank: 0

Full data written to: /Users/barrybrent/data2/run3mar26no4.txt

Found 1 distinct newforms in 0.99 seconds


In [2]:
import pickle
# loads all forms at a given level one at a time converted to lists
def load_newforms_from_file(filename):
    """
    Load all newforms from a file and return them as a list of dictionaries.
    Each dictionary contains the label and coefficients.
    """
    with open(filename, 'r') as f:
        content = f.read()
    
    # Split by the separator line
    blocks = content.split('='*70)
    
    newforms = []
    
    for block in blocks:
        if 'Isogeny class:' in block:
            lines = block.strip().split('\n')
            
            # Extract isogeny class
            isogeny_class = lines[0].replace('Isogeny class:', '').strip()
            
            # Extract representative curve
            rep_curve = lines[1].replace('Representative curve:', '').strip()
            
            # Find the coefficients line
            for i, line in enumerate(lines):
                if line.startswith('['):
                    coeffs = eval(line)
                    break
            
            newforms.append({
                'isogeny_class': isogeny_class,
                'representative': rep_curve,
                'coefficients': coeffs
            })
    
    return newforms

# Load all newforms
newforms = load_newforms_from_file("/Users/barrybrent/data2/run3mar26no4.txt")
print("curve 27a1")
print(f"Loaded {len(newforms)} newforms\n")
print("number of newforms:",len(newforms))
# Access them one at a time
for i, form in enumerate(newforms):
    print(f"Newform {i+1}:")
    print(f"  Isogeny class: {form['isogeny_class']}")
    print(f"  Representative: {form['representative']}")
    print(f"  First 10 coefficients: {form['coefficients'][:10]}")
    print(f"  Total coefficients: {len(form['coefficients'])}")
    print()

# Access individual newforms
newform_1 = newforms[0]['coefficients']  # First newform
#newform_2 = newforms[1]['coefficients']  # Second newform (if it exists)

print(f"Newform 1, coefficient of q^5: {newform_1[4]}")  # Remember: index 4 is a_5
print(newform_1[:500])
curve27a1_list=list(newform_1)
print("length:",len(curve27a1_list))


curve 27a1
Loaded 1 newforms

number of newforms: 1
Newform 1:
  Isogeny class: 27a
  Representative: 27a1
  First 10 coefficients: [1, 0, 0, -2, 0, 0, -1, 0, 0, 0]
  Total coefficients: 2801

Newform 1, coefficient of q^5: 0
[1, 0, 0, -2, 0, 0, -1, 0, 0, 0, 0, 0, 5, 0, 0, 4, 0, 0, -7, 0, 0, 0, 0, 0, -5, 0, 0, 2, 0, 0, -4, 0, 0, 0, 0, 0, 11, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, -6, 0, 0, -10, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, -8, 0, 0, 5, 0, 0, 0, 0, 0, -7, 0, 0, 14, 0, 0, 17, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -5, 0, 0, 0, 0, 0, -19, 0, 0, 10, 0, 0, -13, 0, 0, 0, 0, 0, 2, 0, 0, -4, 0, 0, 0, 0, 0, 0, 0, 0, -11, 0, 0, 8, 0, 0, 20, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 23, 0, 0, 0, 0, 0, 0, 0, 0, -22, 0, 0, -19, 0, 0, 0, 0, 0, 14, 0, 0, 0, 0, 0, -25, 0, 0, 0, 0, 0, 12, 0, 0, -16, 0, 0, 5, 0, 0, 0, 0, 0, -7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 23, 0, 0, 12, 0, 0, 11, 0, 0, 0, 0, 0, 0, 0, 0, 20, 0, 0, -13, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, -28, 0, 0, 0, 0, 0, -22, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 17, 0, 

In [7]:
import pickle
curve27a1_on_primes=[]
index=0
new_index=0
for a in curve27a1_list:
    index+=1
    if is_prime(index):
        new_index+=1
        curve27a1_on_primes.append((new_index,a))
        
with open('/Users/barrybrent/data2/run2mar26no5.txt', 'wb') as wfile:
    pickle.dump(curve27a1_on_primes, wfile)

for pair in curve27a1_on_primes:
    print((pair))

(1, 0)
(2, 0)
(3, 0)
(4, -1)
(5, 0)
(6, 5)
(7, 0)
(8, -7)
(9, 0)
(10, 0)
(11, -4)
(12, 11)
(13, 0)
(14, 8)
(15, 0)
(16, 0)
(17, 0)
(18, -1)
(19, 5)
(20, 0)
(21, -7)
(22, 17)
(23, 0)
(24, 0)
(25, -19)
(26, 0)
(27, -13)
(28, 0)
(29, 2)
(30, 0)
(31, 20)
(32, 0)
(33, 0)
(34, 23)
(35, 0)
(36, -19)
(37, 14)
(38, -25)
(39, 0)
(40, 0)
(41, 0)
(42, -7)
(43, 0)
(44, 23)
(45, 0)
(46, 11)
(47, -13)
(48, -28)
(49, 0)
(50, -22)
(51, 0)
(52, 0)
(53, 17)
(54, 0)
(55, 0)
(56, 0)
(57, 0)
(58, 29)
(59, 26)
(60, 0)
(61, 32)
(62, 0)
(63, -16)
(64, 0)
(65, 35)
(66, 0)
(67, -1)
(68, 5)
(69, 0)
(70, -37)
(71, 0)
(72, 0)
(73, 35)
(74, -13)
(75, 29)
(76, 0)
(77, 0)
(78, -34)
(79, 0)
(80, -31)
(81, 0)
(82, -19)
(83, 0)
(84, 2)
(85, -28)
(86, 0)
(87, 0)
(88, -10)
(89, 0)
(90, 23)
(91, 0)
(92, 0)
(93, -25)
(94, 0)
(95, 32)
(96, 0)
(97, 0)
(98, 0)
(99, -43)
(100, 29)
(101, -1)
(102, 0)
(103, 0)
(104, 0)
(105, -31)
(106, 11)
(107, 0)
(108, 0)
(109, 0)
(110, 26)
(111, -49)
(112, 47)
(113, 0)
(114, 17)
(115, -43)
(116

In [8]:
with open('/Users/barrybrent/data2/run1mar26no9.txt', 'rb') as rfile:
    # from curve 27a1 on primes 1mar26
    curve_twentyseven_prime_list = pickle.load(rfile)
for x in curve_twentyseven_prime_list[:10]:
    print(x)


0
-2
0
0
0
0
0
0
0
0
